In [2]:
!pip install Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.6 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')

path = '/content/drive/MyDrive/Colab Notebooks/Prog/'
datafile = "Dataset1/output.txt"
max_word_len = 25
char_list = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','x','y','z','‘',"ʼ"]
char2idx = {c: i+1 for i, c in enumerate(char_list)}
char2idx['<PAD>'] = 0
idx2char = {v: k for k, v in char2idx.items()}
vocab_size = len(char2idx)

def encode_word(word):
    encoded = [char2idx.get(c, 0) for c in word]
    if len(encoded) < max_word_len:
        encoded += [0] * (max_word_len - len(encoded))
    return encoded[:max_word_len]

def decode_word(indices):
    return ''.join([idx2char.get(i, '') for i in indices if i != 0])

class StemDataset(Dataset):
    def __init__(self, filepath):
        self.data = []
        with open(filepath, 'r', encoding='utf8') as file:
            for line in file:
                line = line.strip()
                if '/' not in line: continue
                stem, affix = line.split('/')
                word = stem + affix
                self.data.append((encode_word(word), encode_word(stem)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x), torch.tensor(y)

class CharCNNBiLSTMStemmer(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, conv_out=128, lstm_hidden=128):
        super(CharCNNBiLSTMStemmer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=embedding_dim, out_channels=conv_out, kernel_size=3, padding=1)
        self.bilstm = nn.LSTM(input_size=conv_out, hidden_size=lstm_hidden, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(lstm_hidden * 2, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        cnn_in = emb.permute(0, 2, 1)
        cnn_out = torch.relu(self.char_cnn(cnn_in))
        cnn_out = cnn_out.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(cnn_out)
        return self.fc(lstm_out)

def train_model(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        outputs = outputs.permute(0, 2, 1)
        loss = criterion(outputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_model(model, dataloader, device):
    model.eval()
    total = 0
    correct = 0
    from Levenshtein import distance as levenshtein_distance
    total_levenshtein = 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            predictions = torch.argmax(outputs, dim=2).cpu().numpy()
            targets = targets.numpy()
            for pred_seq, tgt_seq in zip(predictions, targets):
                pred_word = decode_word(pred_seq)
                tgt_word = decode_word(tgt_seq)
                if pred_word == tgt_word:
                    correct += 1
                total_levenshtein += levenshtein_distance(pred_word, tgt_word)
                total += 1
    acc = 100 * correct / total
    avg_lev = total_levenshtein / total
    return acc, avg_lev

if __name__ == '__main__':
    from sklearn.model_selection import train_test_split
    import matplotlib.pyplot as plt
    import datetime

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    full_dataset = StemDataset(os.path.join(path, datafile))
    indices = list(range(len(full_dataset)))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

    train_set = torch.utils.data.Subset(full_dataset, train_idx)
    test_set = torch.utils.data.Subset(full_dataset, test_idx)

    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

    model = CharCNNBiLSTMStemmer(vocab_size=vocab_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    accs = []
    losses = []
    logs = []
    best_acc = 0

    output_dir = os.path.join(path, 'CharCNN_BiLSTM_Results_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
    os.makedirs(output_dir, exist_ok=True)

    for epoch in range(10):
        loss = train_model(model, train_loader, criterion, optimizer, device)
        acc, lev = evaluate_model(model, test_loader, device)
        accs.append(acc)
        losses.append(loss)
        log_line = f"Epoch {epoch+1} | Loss: {loss:.4f} | Accuracy: {acc:.2f}% | Avg Levenshtein: {lev:.2f}"
        print(log_line)
        logs.append(log_line)
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), os.path.join(output_dir, 'best_model.pt'))

    with open(os.path.join(output_dir, 'training_log.txt'), 'w') as f:
        for line in logs:
            f.write(line + '\n')

    weights = model.fc.weight.detach().cpu().numpy()
    bias = model.fc.bias.detach().cpu().numpy()
    np.savetxt(os.path.join(output_dir, 'final_weights.txt'), weights)
    np.savetxt(os.path.join(output_dir, 'final_bias.txt'), bias)

    model.eval()
    errors = []
    from Levenshtein import distance as levenshtein_distance
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            predictions = torch.argmax(outputs, dim=2).cpu().numpy()
            targets = targets.numpy()
            for pred_seq, tgt_seq in zip(predictions, targets):
                pred_word = decode_word(pred_seq)
                tgt_word = decode_word(tgt_seq)
                if pred_word != tgt_word:
                    errors.append((tgt_word, pred_word))
    with open(os.path.join(output_dir, 'error_samples.txt'), 'w', encoding='utf-8') as f:
        for tgt, pred in errors:
            f.write(f"{tgt}\t{pred}\n")

    plt.figure()
    plt.plot(range(1, len(accs)+1), accs, label='Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.savefig(os.path.join(output_dir, 'accuracy_plot.png'))
    plt.close()

    plt.figure()
    plt.plot(range(1, len(losses)+1), losses, label='Loss', color='red')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(os.path.join(output_dir, 'loss_plot.png'))
    plt.close()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 1 | Loss: 0.5877 | Accuracy: 65.82% | Avg Levenshtein: 0.60
Epoch 2 | Loss: 0.0559 | Accuracy: 74.40% | Avg Levenshtein: 0.45
Epoch 3 | Loss: 0.0403 | Accuracy: 79.15% | Avg Levenshtein: 0.34
Epoch 4 | Loss: 0.0338 | Accuracy: 78.44% | Avg Levenshtein: 0.37
Epoch 5 | Loss: 0.0286 | Accuracy: 83.33% | Avg Levenshtein: 0.29
Epoch 6 | Loss: 0.0251 | Accuracy: 83.76% | Avg Levenshtein: 0.27
Epoch 7 | Loss: 0.0230 | Accuracy: 81.21% | Avg Levenshtein: 0.30
Epoch 8 | Loss: 0.0214 | Accuracy: 81.77% | Avg Levenshtein: 0.29
Epoch 9 | Loss: 0.0184 | Accuracy: 83.97% | Avg Levenshtein: 0.25
Epoch 10 | Loss: 0.0161 | Accuracy: 84.89% | Avg Levenshtein: 0.25
